# Understanding empiric contact data

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/17-empiric-surveys.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Having worked with "fake empiric" matrices under frequency-dependent
transmission in {doc}`16-contact-surveys`, we now turn to real contact-survey
data: the POLYMOD study
([Mossong *et al.*, 2008](https://journals.plos.org/plosmedicine/article?id=10.1371/journal.pmed.0050074)),
whose results are published in matrix form in a supporting table.

That table is **transposed** relative to how we have been thinking about
contact structure: its rows are labelled "age of contact" and its columns "age
group of participant". We think of a vector of prevalence multiplied through a
row of the mixing matrix, so rows must be the participants.
{class}`~summer4.epi.ContactMatrix` fixes that orientation — `values[i, j]` is
the contacts a member of band `i` reports with band `j` — so we transpose once,
on the way in.

The matrix is the POLYMOD Great Britain matrix from Mossong *et al.* (2008),
supporting table
[doi:10.1371/journal.pmed.0050074.st005](https://doi.org/10.1371/journal.pmed.0050074.st005)
(Creative Commons Attribution licence), as transcribed in the source textbook.
summer4 ships no survey data: this page embeds the extract it uses, so nothing
is downloaded when the documentation builds.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from summer4.epi import ContactMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# POLYMOD Great Britain, as published: rows are the AGE OF CONTACT and columns
# the AGE GROUP OF PARTICIPANT.
PUBLISHED = np.array(
    [
        [1.92, 0.65, 0.41, 0.24, 0.46, 0.73, 0.67, 0.83, 0.24, 0.22, 0.36, 0.20, 0.20, 0.26, 0.13],
        [0.95, 6.64, 1.09, 0.73, 0.61, 0.75, 0.95, 1.39, 0.90, 0.16, 0.30, 0.22, 0.50, 0.48, 0.20],
        [0.48, 1.31, 6.85, 1.52, 0.27, 0.31, 0.48, 0.76, 1.00, 0.69, 0.32, 0.44, 0.27, 0.41, 0.33],
        [0.33, 0.34, 1.03, 6.71, 1.58, 0.73, 0.42, 0.56, 0.85, 1.16, 0.70, 0.30, 0.20, 0.48, 0.63],
        [0.45, 0.30, 0.22, 0.93, 2.59, 1.49, 0.75, 0.63, 0.77, 0.87, 0.88, 0.61, 0.53, 0.37, 0.33],
        [0.79, 0.66, 0.44, 0.74, 1.29, 1.83, 0.97, 0.71, 0.74, 0.85, 0.88, 0.87, 0.67, 0.74, 0.33],
        [0.97, 1.07, 0.62, 0.50, 0.88, 1.19, 1.67, 0.89, 1.02, 0.91, 0.92, 0.61, 0.76, 0.63, 0.27],
        [1.02, 0.98, 1.26, 1.09, 0.76, 0.95, 1.53, 1.50, 1.32, 1.09, 0.83, 0.69, 1.02, 0.96, 0.20],
        [0.55, 1.00, 1.14, 0.94, 0.73, 0.88, 0.82, 1.23, 1.35, 1.27, 0.89, 0.67, 0.94, 0.81, 0.80],
        [0.29, 0.54, 0.57, 0.77, 0.97, 0.93, 0.57, 0.80, 1.32, 1.87, 0.61, 0.80, 0.61, 0.59, 0.57],
        [0.33, 0.38, 0.40, 0.41, 0.44, 0.85, 0.60, 0.61, 0.71, 0.95, 0.74, 1.06, 0.59, 0.56, 0.57],
        [0.31, 0.21, 0.25, 0.33, 0.39, 0.53, 0.68, 0.53, 0.55, 0.51, 0.82, 1.17, 0.85, 0.85, 0.33],
        [0.26, 0.25, 0.19, 0.24, 0.19, 0.34, 0.40, 0.39, 0.47, 0.55, 0.41, 0.78, 0.65, 0.85, 0.57],
        [0.09, 0.11, 0.12, 0.20, 0.19, 0.22, 0.13, 0.30, 0.23, 0.13, 0.21, 0.28, 0.36, 0.70, 0.60],
        [0.14, 0.15, 0.21, 0.10, 0.24, 0.17, 0.15, 0.41, 0.50, 0.71, 0.53, 0.76, 0.47, 0.74, 1.47],
    ]
)
AGE_GROUPS = list(range(0, 75, 5))  # 0-4, 5-9, ..., 65-69, 70+
BANDS = (*AGE_GROUPS, math.inf)

polymod = ContactMatrix.from_array(PUBLISHED.T, BANDS)  # transposed on the way in
assert polymod.values[0, 1] == PUBLISHED[1, 0]
polymod

Next, let's see what an empiric matrix looks like.

In [ ]:
matrix_frame = pd.DataFrame(polymod.values, index=AGE_GROUPS, columns=AGE_GROUPS)
matrix_frame.plot(
    kind="imshow",
    title="POLYMOD Great Britain: contacts per participant per day",
    labels={"x": "contact age", "y": "respondent age", "color": "contacts"},
)

In [ ]:
fig = go.Figure(data=[go.Surface(x=AGE_GROUPS, y=AGE_GROUPS, z=polymod.values)])
fig.update_layout(
    title="The same matrix as a surface",
    width=800,
    height=600,
    scene=dict(xaxis_title="contact age", yaxis_title="respondent age", zaxis_title="contacts"),
)
fig

## Matrix characteristics

Several features of this matrix are of epidemiological interest before we put
it anywhere near a model:

- it is broadly **assortative** — the diagonal cells are often the largest, so
  people mix most with people of their own age;
- a second, fainter ridge runs roughly parallel to the diagonal about 30 years
  away from it: mixing between successive generations, parents with their
  children and children with their parents;
- the most intense mixing seems to be among children and young adults, which
  the next section looks at more closely.

There is some jiggle, as with any empirically collected data, and contact
patterns depend heavily on the social characteristics of the place they were
collected in. Even so, this pattern is not unusual; the other POLYMOD countries
showed similar ones.

### Average contacts reported per participant

Next, the total of each row. Each row is a respondent age band — the
susceptible band at risk of infection — so its total is the average number of
contacts reported by participants in that band:
{meth}`~summer4.epi.ContactMatrix.total`. Older children and young adults
report the most contacts, with a gradual decline through the rest of adulthood.
Remember these are simply averages of what survey participants reported.

In [ ]:
row_totals = pd.Series(polymod.total(), index=pd.Index(AGE_GROUPS, name="age"))
assert row_totals.idxmax() in (10, 15)
pd.DataFrame(polymod.values, index=AGE_GROUPS, columns=AGE_GROUPS).plot.area(
    title="Contacts reported per participant, by the age of the contact",
    labels={"index": "respondent age", "value": "contacts", "variable": "contact age"},
)

### Age of contacts reported by all participants

Summing over the columns instead gives a largely similar pattern, for similar
reasons. But there is one noticeable difference: an uptick for the oldest band.
Random variability? The rate nearly doubles from the 65–69 band to the 70+
band, and it does so for many respondent ages — which is the same observation as
noticing that the right-most column of the heatmap is a little brighter than
the one to its left.

There is a good reason for it. Summing over columns considers a quite
different quantity, and it illustrates the point from
{doc}`16-contact-surveys`: the number of reported contacts is partly determined
by how many people in a band are available to be contacted. For these bands in
a high-income country the last band should be larger than the second last,
because 65–69 covers five years while "70 and over" covers everyone older.
Even if that group is less socially active, in many high-income settings it is
at least twice the size of the band below it. Since the survey records contacts
with *anyone* from a band, we expect more reported contacts with larger bands.

In [ ]:
column_totals = pd.Series(polymod.values.sum(axis=0), index=pd.Index(AGE_GROUPS, name="age"))
assert column_totals[70] > 1.5 * column_totals[65]
assert row_totals[70] < row_totals[65]
pd.DataFrame(polymod.values.T, index=AGE_GROUPS, columns=AGE_GROUPS).plot.area(
    title="Times each age was reported as a contact, by respondent age",
    labels={"index": "contact age", "value": "contacts", "variable": "respondent age"},
)

This should reinforce the point that we have to think carefully when taking a
matrix from one setting to another, even if we want to keep its contact
structure essentially the same. {doc}`19-adapting-mixing-matrices` does exactly
that.